# Capstone Project — End-to-End Protein ML Pipeline
## Integrating All 17 Modules

## TL;DR — Plain English
**What this notebook does:** Connects everything you learned into one complete pipeline:
sequence → structure prediction → confidence assessment → variant effect → ΔΔG fine-tuning → deployment.

**This is the interview simulation.** A real ML engineer at computational biology ML teams does exactly this:
given a protein of interest, run the full computational pipeline to inform drug design.

**Time:** 4-6 hours | **Prerequisites:** All modules 00-16


## Capstone — Quick Reference for Beginners

| Term | Plain English |
|------|---------------|
| **EGFR** | Epidermal Growth Factor Receptor — a kinase mutated in ~15% of lung cancers; targeted by erlotinib |
| **kinase** | Enzyme that adds phosphate groups to other proteins — switches them on or off |
| **L858R** | Most common EGFR activating mutation; leucine (L) at position 858 changed to arginine (R) |
| **T790M** | EGFR resistance mutation — emerges after first-generation inhibitor treatment |
| **ESM-2** | Meta's protein language model — converts amino acid sequence to rich embedding vectors |
| **delta-delta-G (ddG)** | Free energy change of mutation: negative = stabilising, positive = destabilising |
| **mutation scanning** | Test all 19 substitutions at every position — identifies hot-spot residues |
| **conformal prediction** | Gives guaranteed-coverage prediction intervals (e.g. 95% CI on ddG) |
| **Bayesian optimisation** | Efficient search strategy: model uncertainty -> pick most informative next experiment |
| **pLDDT** | AlphaFold's confidence score per residue (0-100) — low = disordered or unreliable |
| **FastAPI endpoint** | Web URL that accepts a mutation and returns a ddG prediction in real time |
| **executive summary** | 200-word plain-English description of your findings written for a non-technical audience |

## 🟢 How to Use This Capstone Notebook

**This is the FINAL notebook.** If you've done all 17 modules, run each cell top to bottom — you should recognise every function from a previous module.

If you haven't completed all modules yet, that's OK — treat each step as a **preview** and come back after completing the prerequisites listed in the pipeline diagram below.

| Step | Prerequisite modules |
|------|---------------------|
| 1. Sequence validation | 00, 01 |
| 2. Domain analysis | 01 |
| 3. ESM-2 embeddings | 15 |
| 4. Structure confidence | 07/03 |
| 5. ΔΔG prediction | 10/01 |
| 6. Uncertainty | 13/01 |

> **Interview readiness:** After running this notebook end-to-end, time yourself explaining the pipeline in 3 minutes. If you can do that without notes, you are ready.

---
## The Pipeline You Will Build

```
INPUT: Protein sequence (human EGFR kinase domain, a cancer drug target)
           │
           ▼
    Step 1: Parse & validate sequence (Module 00/01)
           │
           ▼
    Step 2: Sequence analysis — find domains, signal peptides (Module 01)
           │
           ▼
    Step 3: Build ML features — ESM-2 embeddings (Module 15)
           │
           ▼
    Step 4: Structure prediction confidence (Module 07/03)
           │
           ▼
    Step 5: ΔΔG prediction for known mutations (Module 10/01)
           │
           ▼
    Step 6: Uncertainty quantification — which predictions to trust? (Module 13)
           │
           ▼
OUTPUT: Ranked list of mutations with predicted binding effect + confidence
```

Each step uses concepts and code from the corresponding module.


## 🟢 Complete Beginner's Guide

**Assumed background:** Has completed all 17 modules. Now ready to integrate everything into a single pipeline.

### What this notebook does

This capstone builds an end-to-end EGFR mutation effect prediction pipeline that uses:
- Module 00: Python/data loading
- Module 01–02: Sequence parsing, variant annotation
- Module 03: Structural analysis, RMSD computation
- Module 05–06: Deep learning and GNN-based features
- Module 07/10: AlphaFold3/OpenFold structure prediction integration
- Module 13: Uncertainty quantification on predictions
- Module 16: MLOps — logging, monitoring, deployment

### How to approach this notebook

Think of it as a **systems integration test**. Each section calls a component you built in a previous module. Your job here is not to learn new concepts — it is to see how they compose into a working system.

### What to do if a step fails

1. **Read the error message fully** — it tells you which module is breaking.
2. **Go back to that module's notebook** — re-run it standalone to verify it works.
3. **Check data shapes** — most failures are shape mismatches between pipeline stages.
4. **Use the fallback data** — each section has a simulated fallback; use it to continue and come back to fix the real data later.
5. **Isolate the failing cell** — run sections before and after the failure to narrow down where the break is.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `EGFR` | Epidermal Growth Factor Receptor — a kinase mutated in many lung cancers |
| `ΔΔG` | Change in binding free energy caused by a mutation |
| `pipeline` | A sequence of processing steps where output of one feeds next |
| `end-to-end` | From raw input (sequence/mutation) to final prediction (effect score) |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — draw the pipeline diagram on paper.
2. **Run code cells one at a time** — print the data shape at each stage transition.
3. **Modify one mutation and re-run** — change the EGFR variant and trace how all downstream predictions change.

### If you're stuck

- **YouTube:** 'How to structure a machine learning project' by Full Stack Deep Learning.
- **Book:** *Designing Machine Learning Systems* by Chip Huyen — Chapter on ML pipelines.
- **Web:** https://clinicalgenome.org/affiliation/50013/ — EGFR variant curation resources.


In [ ]:
# STEP 1: Sequence Validation and Analysis
# From Module 00/01 (Python Core for Bioinformatics) + Module 01/01 (Alignment)

import numpy as np
import torch

# EGFR kinase domain (real sequence, truncated to 50 aa for demo)
# UniProt P00533, residues 712-761
EGFR_KINASE = "KVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLSATSNNSTVACIDRNGLQSCPIKEDSFLQRYSSDPTGALTEDSIDDTFLPVPEYINQSVPKRPAGSVQNPVYHNQPLNPAPSRDPHYQDPHSTAVGNPEYLNTVQPTCVNSTFDSPAHWAQKGSHQISLDNPDYQQDFFPKEAKPNGIFKGSTAENAEYLRVAPQSSEFIGA"
EGFR_SHORT  = EGFR_KINASE[:50]

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_PROPS = {
    'hydrophobic': set('AILMFWVP'),
    'polar':       set('STNQ'),
    'charged_pos': set('KRH'),
    'charged_neg': set('DE'),
    'aromatic':    set('FWY'),
}

def validate_sequence(seq):
    invalid = set(seq) - set(AMINO_ACIDS)
    if invalid:
        return False, f"Invalid amino acids: {invalid}"
    return True, f"Valid: {len(seq)} residues"

def analyze_composition(seq):
    counts = {aa: seq.count(aa)/len(seq) for aa in AMINO_ACIDS}
    # Group properties
    result = {}
    for prop, aas in AA_PROPS.items():
        result[prop] = sum(counts.get(aa, 0) for aa in aas)
    result['mw_estimate_kDa'] = len(seq) * 110 / 1000  # ~110 Da per residue
    return result

valid, msg = validate_sequence(EGFR_SHORT)
comp = analyze_composition(EGFR_SHORT)
print(f"Sequence: {EGFR_SHORT}")
print(f"Validation: {msg}")
print(f"Composition:")
for k, v in comp.items():
    print(f"  {k:<20}: {v:.3f}")

> 📋 **Expected output:** The EGFR protein sequence (~1210 amino acids) starting with `MRPSGTAGAA...` fetched from UniProt accession P00533.
> ✅ If you see the sequence printed, your internet connection and UniProt API are working.
> ❌ If you see a timeout error, check your internet or try again in 30 seconds.